In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.schema import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain.llms.base import LLM
from langchain_deepseek import ChatDeepSeek

from typing import Optional, List, Dict, Any, Tuple
import requests

from dataclasses import dataclass, field
import pickle

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from tqdm.auto import tqdm

import os

import pickle

from llm_core.llm_core import deepseek, chatgpt, giga

from langchain.text_splitter import RecursiveCharacterTextSplitter
import tiktoken

from langchain_core.documents.base import Document

D:\Anaconda\envs\scenario\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Разбить модели

In [2]:
llm_type = "deepseek"

if llm_type == "giga":
    llm = giga
elif llm_type == "chatgpt":
    llm = chatgpt
elif llm_type == "deepseek":
    llm = deepseek

In [3]:
def calculate_tokens(text: str) -> int:
    """Функция для точного подсчета токенов с помощью tiktoken"""
    encoder = tiktoken.get_encoding("cl100k_base")
    return len(encoder.encode(text))

def split_text(text: str) -> list[str]:
    """Разбивает текст на чанки с учетом ограничений токенов"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,
        chunk_overlap=0,
        length_function=calculate_tokens,
        separators=["\n\n", "\n", " ", ""] 
    )
    
    return text_splitter.split_text(text)

In [4]:
summaries_folder = "summaries"
book_name = "data/Ten-i-Plama.txt"

In [5]:
with open(book_name, 'r', encoding='utf-8') as file:
  book = file.read()

In [6]:
summaries = os.listdir(summaries_folder)
sum_texts = []
for sum_file in tqdm(summaries):
    if sum_file.endswith(".txt"):
        sum_num = sum_file.replace(".txt", "")
        with open(os.path.join(summaries_folder, sum_file), "r") as file:
            sum_text = file.read()
        sum_texts.append(Document(metadata = {"source_id": sum_num}, page_content=sum_text))

# Добавить в метаданные суммаризацию
text_splitter = RecursiveCharacterTextSplitter(
    separators=["========== ", "***"],
)
texts = text_splitter.create_documents([book])
for num, text in enumerate(texts):
    text.metadata = {"source_id": num, 
                     # "summarization": sum_texts[num].page_content
                    }
    
texts.extend(sum_texts)

100%|██████████████████████████████████████████████████████████████████████████████| 155/155 [00:00<00:00, 5000.79it/s]


In [7]:
texts_splitted = []
for text in tqdm(texts):
    metadata = text.metadata
    chunks = split_text(text.page_content)
    for chunk in chunks:
        texts_splitted.append(Document(metadata=metadata, page_content=chunk))

100%|███████████████████████████████████████████████████████████████████████████████| 310/310 [00:02<00:00, 113.47it/s]


In [8]:
import pickle

In [9]:
with open("data/langchain_chunks.pkl", "wb") as file:
    pickle.dump(texts_splitted, file)

# Добавить все это в ElasticSearch

In [1]:
from vector_store.vector_store import VectorStore
from vector_store.elastic_search import ElasticStore
import utils.settings as settings
import pickle
from itertools import islice

In [2]:
with open("data/langchain_chunks.pkl", "rb") as file:
    documents = pickle.load(file)

In [3]:
# doc_iter = iter(documents)
# batch_size=100
# while True:
#     batch = list(islice(doc_iter, batch_size))
#     if not batch:
#         break
#     print(f"Added {len(batch)} documents")

In [4]:
settings.ES_PICKLE_DOCUMENTS_PATH

'data\\langchain_chunks.pkl'

In [5]:
settings.ES_BATCH_SIZE

100

In [6]:
retriever = ElasticStore(force_reload=True)

2025-05-11 12:18:01,898 - elastic_transport.transport - INFO - GET http://localhost:9200/ [status:200 duration:0.029s]
2025-05-11 12:18:01,906 - elastic_transport.transport - INFO - HEAD http://localhost:9200/shadow_and_flame_ [status:200 duration:0.004s]
2025-05-11 12:18:02,356 - elastic_transport.transport - INFO - DELETE http://localhost:9200/shadow_and_flame_ [status:200 duration:0.447s]
2025-05-11 12:18:02,396 - elastic_transport.transport - INFO - HEAD http://localhost:9200/shadow_and_flame_ [status:404 duration:0.004s]
2025-05-11 12:18:03,013 - httpx - INFO - HTTP Request: POST https://api.proxyapi.ru/openai/v1/embeddings "HTTP/1.1 200 OK"
2025-05-11 12:18:05,871 - elastic_transport.transport - INFO - PUT http://localhost:9200/shadow_and_flame_ [status:200 duration:2.586s]
2025-05-11 12:18:07,520 - httpx - INFO - HTTP Request: POST https://api.proxyapi.ru/openai/v1/embeddings "HTTP/1.1 200 OK"
2025-05-11 12:18:12,757 - elastic_transport.transport - INFO - PUT http://localhost:92

In [147]:
llm = deepseek

In [197]:
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

from tqdm.auto import tqdm

In [198]:
JsonOutputParser

langchain_core.output_parsers.json.JsonOutputParser

In [199]:
# done
from abc import ABC, abstractmethod
from typing import Any, Dict, Optional, Union
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable
from langchain_core.language_models import BaseLanguageModel
from langchain_core.output_parsers import BaseOutputParser

class LLMBase(ABC):
    """Abstract base class for LLM-powered agents with sync/async support."""
    
    def __init__(self, llm: BaseLanguageModel, 
                 system_prompt: str, 
                 parser: Optional[BaseOutputParser] = None):
        """
        Args:
            llm: Initialized language model
            system_prompt: Base system prompt template
        """
        self.llm = llm
        self._system_prompt = system_prompt
        self._prompt_template: Optional[ChatPromptTemplate] = None
        self._parser = parser

    @property
    def prompt_template(self) -> ChatPromptTemplate:
        """Cached prompt template property"""
        if self._prompt_template is None:
            self._prompt_template = self._create_prompt_template()
        return self._prompt_template

    def _create_prompt_template(self) -> ChatPromptTemplate:
        """Construct prompt template from system message"""
        return ChatPromptTemplate.from_messages([
            ("system", self._system_prompt),
            ("placeholder", "{messages}")
        ])

    def make_llm_chain(self) -> Runnable:
        """Create runnable LLM chain with prompt template"""
        if self._parser:
            return self.prompt_template | self.llm | self._parser
        else:
            return self.prompt_template | self.llm

    @abstractmethod
    def make_user_prompt(self, **kwargs: Dict[str, Any]) -> Dict[str, Any]:
        """Abstract method to format user input for LLM"""
        raise NotImplementedError

    def invoke(self, **kwargs: Dict[str, Any]) -> str:
        """Execute LLM chain synchronously"""
        chain_input = self.make_user_prompt(**kwargs)
        result = self.make_llm_chain().invoke(chain_input)
        return self._process_output(result)

    async def ainvoke(self, **kwargs: Dict[str, Any]) -> str:
        """Execute LLM chain asynchronously"""
        chain_input = self.make_user_prompt(**kwargs)
        result = await self.make_llm_chain().ainvoke(chain_input)
        return self._process_output(result)

    def _process_output(self, output: Any) -> str:
        """Uniform output processing"""
        if hasattr(output, 'content'):
            return output.content
        return output

    def __call__(self, **kwargs: Dict[str, Any]) -> Union[str, Any]:
        """Alias for invoke"""
        return self.invoke(**kwargs)

In [230]:
# done
reasoning_prompt = """Ты — аналитический агент-рассуждатель. Твоя задача — критически оценить собранную информацию и принять решение о дальнейших действиях.

## Твои входные данные: 
1. Исходный вопрос пользователя: str //Исходный вопрос, который задал пользователь
2. Собранный контекст: List[str] // Уже имеющийся у тебя контекст

# Инструкции:
1. Проанализируй контекст на соответствие исходному вопросу
2. Оцени полноту информации по критериям:
   - Покрыты ли все аспекты вопроса?
   - Есть ли противоречия в данных?
   - Присутствуют ли неясные моменты?
   - Какой информации не хватает, чтобы дать полный, четкий и ясный ответ на вопрос?
3. Прими решение по следующей логике:
   * Если контекст НЕ ПОЛНОСТЬЮ отвечает на вопрос -> SEARCH
   * Если контекст ДОСТАТОЧЕН и НЕТ ПРОТИВОРЕЧИЙ -> ANSWER
4. Если ты понимаешь, что для исчерпывающего ответа на вопрос требуется больше информации, напиши, какая дополнительная информация тебе нужна. 
   - Опирайся на контекст, задавай вопрос и делай описание исходя из той контекстной информации, которая уже есть
   - Постарайся давать конкретные задачи по сбору информации, избегай слишком расплывчатых задач.

# Выходные данные: 
Ты должен вернуть JSON со следующими полями: 

{{"reasoning": str, // Твои детальные рассуждения
  "next_step": str, // Если ты считаешь, что требуется собрать еще какую-то информацию, чтобы ответить на вопрос, верни SEARCH. Если ты считаешь, что информации достаточно для ответа, верни ANSWER.
  "to_collect": str // Описание задания для сбора информации следующему агенту - какую именно информацию требуется собрать?
}}

"""

class ReasoningAgent(LLMBase):
    
    def make_user_prompt(self, user_question, context):
        message = [f"**Исходный вопрос пользователя:** {user_question}", 
                   f"**Собранный контекст:** {context}"]
        user_prompt = "\n\n".join(message)
        messages = {"messages": [("user", user_prompt)]}
        return messages

reasoning_agent = ReasoningAgent(llm, reasoning_prompt, JsonOutputParser())


In [201]:
# additional_questions = reasoning_agent.invoke(user_question = "Какие отношения между Малекитом и Нерданель?", context=[])

In [202]:
# additional_questions

In [203]:
# done

question_generator_prompt = """Ты — профессиональный генератор поисковых запросов для RAG системы. 

# Задача:
Сгенерируй до 5 конкретных вопросов на основе:
1. Исходного запроса пользователя: str // Исходный вопрос, который задал пользователь
2. Контекста размышлений агента: str // Инструкция, по которой ты должен сгенерировать вопросы.

# Требования к вопросам:
1. Максимально конкретные и узкие формулировки
2. Использование терминов из контекста размышлений
3. Формат как для поисковой системы (без предложений-ответов)
4. Каждый вопрос должен уточнять разные аспекты
5. Избегай общих вопросов в стиле "расскажи всё о..."
6. Вопрос должен быть К СЮЖЕТУ книги, к персонажам. Тебе запрещено использовать вопросы формата "Роль Сони в сюжете".

# Правила вывода:
- Только JSON-объект с массивом "questions"
- Не более 10 вопросов
- Каждый вопрос в кавычках
- Без Markdown-разметки
- Используй точные термины из контекста

Сгенерируй вопросы:"""

class QuestionAgent(LLMBase):
    
    def make_user_prompt(self, user_question, reasoning):
        message = [f"**Исходный вопрос пользователя:** {user_question}", 
                   f"**Размешления агента:** {reasoning}"]
        user_prompt = "\n\n".join(message)
        messages = {"messages": [("user", user_prompt)]}
        return messages

questions_agent = QuestionAgent(llm, question_generator_prompt, JsonOutputParser())

In [204]:
# additional_questions

In [205]:
# question_list = questions_agent.invoke(user_question="Какие отношения были у Лаурэфинде с его отцом?", 
#                                        reasoning=additional_questions['reasoning'])

In [206]:
# question_list

In [207]:
# done
from llm_core.llm_core import deepseek, chatgpt, giga
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.retrievers import BaseRetriever

retrieve_prompt = (
    "Ответь на заданный тебе вопрос, используя данный контекст." 
    "Извлеки из контекста ТОЛЬКО ту информацию, которая имеет отношение к изначальному вопросу, игнорируй все лишнее."
    "Если ты не можешь извлечь ответ на вопрос из контекста, честно напиши, что не знаешь. "
    "Контекст: {context}"
) 

retr = retriever._retriever

class RetrieveAgent(LLMBase):

    def __init__(self, llm: BaseLanguageModel, 
                 system_prompt: str, 
                 retriever: BaseRetriever,
                 parser: Optional[BaseOutputParser] = None):
        """
        Args:
            llm: Initialized language model
            system_prompt: Base system prompt template
        """
        self.llm = llm
        self._system_prompt = system_prompt
        self._retriever = retriever
        self._prompt_template: Optional[ChatPromptTemplate] = None
        self._parser = parser

    def make_llm_chain(self) -> Runnable:
        """Create runnable LLM chain with prompt template"""
        question_answer_chain = create_stuff_documents_chain(self.llm, self.prompt_template)
        chain = create_retrieval_chain(self._retriever, question_answer_chain)
        return chain
    
    def make_user_prompt(self, query):
        return {"input": query}

retrieve_agent = RetrieveAgent(llm, retrieve_prompt, retriever=retr)

In [208]:
# retrieve_agent.invoke(query="Какие отношения были между Лаурэфиндэ и его отцом?")

In [209]:
# done

final_answer_prompt = """Ты — эксперт-аналитик, формирующий итоговый ответ. 

# ЗАДАЧА:
Синтезировать ответ ИСКЛЮЧИТЕЛЬНО на основе:
1. Исходный вопрос пользователя: str //Исходный вопрос, который задал пользователь
2. Цепочка размышлений: List[str] // 
3. Собранный контекст: List[str] // Уже имеющийся у тебя контекст

# ИНСТРУКЦИИ:
1. Анализ:
   - Сопоставь каждый элемент контекста с запросом
   - Выяви ключевые доказательные факты
   - Опирайся только на ту информацию в контексте, которая имеет отношение к исходному запросу пользователя, не используй нерелевантную информацию

2. Формирование ответа:
   - Структурируй ответ: тезис → доказательства → вывод
   - Используй ТОЧНЫЕ цитаты из контекста
   - Сохрани цепочку логических умозаключений

3. Контроль качества:
   - Если информации НЕДОСТАТОЧНО → явно укажи это
   - Запрещены домыслы/интерпретации
   - Ответ должен покрывать ВСЕ аспекты запроса

# ПРИМЕРЫ:
Запрос: "Каковы мотивы главного героя в романе X?"
Правильный ответ: {{
  "final_answer": "Согласно анализу главных диалогов (с. 45-47) и авторским комментариям (с. 112):\n1. Основной мотив - ...\n2. Второстепенный мотив - ...\nВывод: Совокупность факторов указывает на..." 
}}

Недопустимый ответ: "Герой хотел добиться успеха, возможно из-за детских травм" 

# ТРЕБОВАНИЯ К ВЫВОДУ:
- ТОЛЬКО JSON-объект с ключом "final_answer"
- Ответ на языке оригинала запроса
- Четкая структура с указанием источников
- Минимум 3 доказательных пункта при наличии данных
- Объем: 150-300 слов

Сформируй ответ:"""


class AnswerAgent(LLMBase):
    
    def make_user_prompt(self, user_question, thoughts, context):
        message = [f"**Исходный вопрос пользователя:** {user_question}", 
                   f"**Цепочка размышлений:** {thoughts}", 
                   f"**Собранный контекст:** {context}"]
        user_prompt = "\n\n".join(message)
        messages = {"messages": [("user", user_prompt)]}
        return messages

answer_agent = AnswerAgent(llm, final_answer_prompt, JsonOutputParser())

In [210]:
# answer_agent.invoke(user_question="Какие отношения были между Лаурэфиндэ и его отцом?", 
#                     thoughts=[], 
#                     context=["""Вопрос: Какие отношения были у Лаурэфиндэ с его отцом Аркуэнвилом?  \n\nОтвет: Отношения Лаурэфиндэ с отцом были сложными. Аркуэнвил пытался навязать сыну традиции Ваниар, что вызывало конфликты, особенно после ухода матери Лаурэфиндэ, которая отказалась подчиняться воле мужа. Однако со временем отец стал идти на уступки, предлагая сыну символы его принадлежности к Нолдор, но Лаурэфиндэ оставался сдержанным и недоверчивым, помня прошлые разногласия."""])

In [211]:
# done

from langgraph.graph import StateGraph, START, END
# Определяем структуру состояния агента
class AgentState(TypedDict):
    user_question: str
    thoughts: List[str]
    context: List[Dict[str, Any]]
    questions: List[str]
    to_collect: str
    final_answer: str
    next_step: str
    max_n_iterations: int
    n_iteration: int

In [222]:
reasoning_agent = ReasoningAgent(llm, reasoning_prompt, JsonOutputParser())
questions_agent = QuestionAgent(llm, question_generator_prompt, JsonOutputParser())
retrieve_agent = RetrieveAgent(llm, retrieve_prompt, retriever=retr)
answer_agent = AnswerAgent(llm, final_answer_prompt, JsonOutputParser())

default_values = [
                    ("max_n_iterations", 10),
                    ("n_iteration", 0),
                    ("user_question", ""),
                    ("thoughts", []), 
                    ("context", []), 
                    ("questions", []),
                    ("final_answer", "")
                ]

def reasoning_node(state: AgentState):
    for key, value in default_values:
        if key not in state:
            state[key] = value
        
    additional_questions = reasoning_agent.invoke(user_question=state['user_question'], 
                                                  context=state['context'])
    state['thoughts'].append(additional_questions['reasoning'])
    state['to_collect'] = additional_questions['to_collect']
    state['next_step'] = additional_questions['next_step']
    state['n_iteration'] += 1
    print(additional_questions)
    return state

def cond_edge_reasoner(state: AgentState):
    if state['next_step'] == "SEARCH" and state['n_iteration'] <= state['max_n_iterations']:
        return 'search_node'
    else:
        return 'final_answer_node'

def search_node(state: AgentState):
    questions = questions_agent.invoke(user_question=state['user_question'], 
                                       reasoning=state['to_collect'])['questions']
    for query in questions:
        print(query)
        ans = retrieve_agent.invoke(query=query)
        state['context'].append({query: ans['answer']})
    return state

def final_answer_node(state: AgentState):
    final_answer = answer_agent.invoke(user_question=state['user_question'], 
                        thoughts=state['thoughts'], 
                        context=state['context'])
    state['final_answer'] = final_answer
    return state

In [223]:
workflow = StateGraph(AgentState)

workflow.add_node("reasoning_node", reasoning_node)
workflow.add_node("search_node", search_node)
workflow.add_node("final_answer_node", final_answer_node)

workflow.add_edge(START, "reasoning_node")
workflow.add_conditional_edges("reasoning_node", cond_edge_reasoner, 
                               ["search_node", "final_answer_node"])
workflow.add_edge("search_node", "reasoning_node")
workflow.add_edge("final_answer_node", END)

app = workflow.compile()

In [1]:
# inputs = {"user_question": "Какие отношения между Малекитом и Нерданель?"}
# state_res = app.invoke(inputs)

In [2]:
# print(state_res['final_answer']['final_answer'])